## [Link github repo](https://github.com/CaoAnhVu141/metro-lbs)

In [ ]:
%run ./dev-v1.ipynb

## Cầu nối tên hàm giữa Khối 1-3 và Khối 4

TV1 đặt tên hàm theo snake_case (`random_state`, `get_neighbors`), còn Khối 4
gọi theo camelCase (`randomState`, `getNeighbors`). Cell dưới gán tên mới cho
cùng một hàm, không sao chép hay sửa gì code của TV1.

Hai tên trỏ về chung một hàm, nên sau này TV1 sửa `dev-v1.ipynb` thì Khối 4
tự động dùng bản mới.

In [ ]:
# ====== CẦU NỐI sang Khối 1-3 của TV1 ======
# TV1 dùng snake_case, Khối 4 gọi camelCase. Gán tên mới cho cùng một hàm.

randomState = random_state
getNeighbors = get_neighbors
randomNeighbor = random_neighbor

## 4.1 localBeamSearch

Giữ beamWidth phương án song song. Mỗi vòng chỉnh nhẹ tất cả phương án
đang có, gom chung một rổ, rồi giữ lại beamWidth phương án điểm cao nhất.

beamWidth = 1 chính là Hill Climbing.


In [ ]:
def localBeamSearch(beamWidth,maxRounds=100,maxNoGain=10, seed=0,numStations=MAX_NEW_STATIONS, weights=DEFAULT_WEIGHTS):
    rng = np.random.default_rng(seed)
    # tạo phương án ngẫu nhiên làm điểm xuất phát
    beam = [randomState(numStations,rng) for _ in range(beamWidth)]
    beamScores = np.array([objective(state,weights) for state in beam])
    
    # ghi dữ liệu ban đầu
    bestIndex = int(np.argmax(beamScores))
    bestState = beam[bestIndex]
    bestScore = float(beamScores[bestIndex])
    
    history = [{
        "round": 0,
        "bestScore": bestScore,
        "beamBest": float(beamScores.max()),
        "beamMean": float(beamScores.mean()),
    }]
    noGainRounds = 0
    
    # với phương án đang có, chỉnh nhẹ 1 trạm dời sang trạm khác 
    for roundNo in range(1,maxRounds+1):
        neighborPool = list({
            newState
            for state in beam
            for newState in getNeighbors(state)
        })
        
        if not neighborPool:
            break
        
        neighborScores = np.array([objective(state,weights) for state in neighborPool])
        topIndexs = np.argsort(neighborScores)[::-1][:beamWidth]
        beam = [neighborPool[i] for i in topIndexs]
        beamScores = neighborScores[topIndexs]
        
        # beamScores[0] tốt nhất của riêng vòng này
        # bestScore tốt nhất từ đầu tới giờ
        if beamScores[0] > bestScore:
            bestState = beam[0]
            bestScore = float(beamScores[0])
            noGainRounds = 0
        else:
            noGainRounds +=1 
        history.append({
            "round": roundNo,
            "bestScore": bestScore,
            "beamBest": float(beamScores[0]),
            "beamMean": float(beamScores.mean()),
        })
        
        # không được thì dừng lại việc thực thi
        if noGainRounds >= maxNoGain:
            break
        
    return bestState, bestScore, history
    


## 4.2 stochasticLocalBeamSearch

Lấy top-k tuyệt đối thì cả `beamWidth` suất thường rơi hết vào con cháu của
cùng một phương án. Beam mất đa dạng, `beamWidth` phương án gần như giống hệt
nhau, thuật toán thoái hoá thành Hill Climbing.

Bốc thăm thì phương án điểm thấp vẫn có cơ hội sống sót. Một cách bố trí trạm
tạm thời kém, sau vài vòng chỉnh sửa, có thể vượt lên dẫn đầu.

In [ ]:
def stochasticLocalBeamSearch(beamWidth, maxRounds=100, maxNoGain=10, seed=0,
                              numStations=MAX_NEW_STATIONS, weights=DEFAULT_WEIGHTS):
    rng = np.random.default_rng(seed)
    
    # bốc đại 1 beamWidth phương án để có chỗ bắt đầu
    beam = [randomState(numStations,rng) for _ in range(beamWidth)]
    beamScores = np.array([objective(state,weights) for state in beam])
    
    bestIndex = int(np.argmax(beamScores))
    bestState = beam[bestIndex]
    bestScore = float(beamScores[bestIndex])
    
    history = [{
        "round": 0,
        "bestScore": bestScore,
        "beamBest": float(beamScores.max()),
        "beamMean": float(beamScores.mean()),
    }]

    noGainRounds = 0
    
    for roundNo in range(1,maxRounds+1):
        # gom vào 1 rổ chung
        neighborPool = list({
            newState
            for state in beam
            for newState in getNeighbors(state)
        })

        if not neighborPool:
            break
        
        neighborScores = np.array([objective(state, weights) for state in neighborPool])

        # Phát vé theo thứ hạng thay vì theo điểm thô: tốt nhất hạng 1, kế tiếp hạng 2...
        # Số vé = 1/hạng. Không phụ thuộc thang điểm nên vẫn giữ được sức ép chọn lọc
        # kể cả khi điểm các phương án chênh nhau rất xa do bị phạt nặng.
        order = np.argsort(neighborScores)[::-1]
        ranks = np.empty(len(neighborScores))
        ranks[order] = np.arange(1, len(neighborScores) + 1)
        tickets = 1.0 / ranks
        pickProbs = tickets / tickets.sum()
        
        # lấy beamWidth phương thức khác nhau
        pickedIndexes = rng.choice(len(neighborPool), size=beamWidth, replace=False, p=pickProbs)
        beam = [neighborPool[i] for i in pickedIndexes]
        beamScores = neighborScores[pickedIndexes]
        
        # dùng argMax để tìm phương án tốt nhất của vòng
        roundBestIndex = int(np.argmax(beamScores))
        roundBestScore = float(beamScores[roundBestIndex])
        
        if roundBestScore > bestScore:
            bestState = beam[roundBestIndex]
            bestScore = roundBestScore
            noGainRounds = 0
        else:
            noGainRounds += 1

        history.append({
            "round": roundNo,
            "bestScore": bestScore,
            "beamBest": roundBestScore,
            "beamMean": float(beamScores.mean()),
        })
        
        # thực hiện dùng nếu k có kết quả
        if noGainRounds >= maxNoGain:
            break
        
    return bestState, bestScore, history
        
        

## 4.3 randomRestartBeamSearch


In [ ]:
def randomRestartBeamSearch(numRestarts, beamWidth, maxRounds=20, maxNoGain=10,
                            seed=0, numStations=MAX_NEW_STATIONS, weights=DEFAULT_WEIGHTS,
                            searchFn=localBeamSearch):
    bestState = None
    bestScore = -np.inf
    history = []
    
    # Mỗi lần chạy phải đổi seed, không thì bốc trúng y hệt bộ phương án cũ
    # và chạy lại numRestarts lần cũng chỉ ra đúng một kết quả
    for restartNo  in range(numRestarts):
        state, score, runHistory = searchFn(
            beamWidth,
            maxRounds=maxRounds,
            maxNoGain=maxNoGain,
            seed=seed + restartNo,
            numStations=numStations,
            weights=weights,
        )
        
        # Gắn thêm số thứ tự lần chạy vào nhật ký, để Khối 6 tách ra vẽ riêng
        # từng lần và thấy được lần nào ăn may lần nào xui
        for row in runHistory:
            history.append({"restart": restartNo, **row})

        # Giữ lại phương án tốt nhất trong tất cả các lần chạy
        if score > bestScore:
            bestState = state
            bestScore = score

    return bestState, bestScore, history

# Khối 5. Thực nghiệm
Chạy các thuật toán ở Khối 4 trên dữ liệu thật, thu số liệu cho chương 4 báo cáo.

## 5.1 runExperiment — hàm chạy thí nghiệm dùng chung

Ba mục 5.2, 5.3, 5.4 thực chất là cùng một việc, chỉ khác thứ đem ra quét:

| Mục | Quét cái gì |
|---|---|
| 5.2 | `beamWidth` = 1, 3, 5, 10, 20 |
| 5.3 | Thuật toán = local beam, stochastic, random restart |
| 5.4 | Trọng số dân số / chi phí |


| Tên | Ý nghĩa |
|---|---|
| `searchFn` | Thuật toán đem chạy: `localBeamSearch`, `stochasticLocalBeamSearch`... |
| `numSeeds` | Chạy bao nhiêu lần, mặc định 20 |
| `label` | Tên cấu hình, để phân biệt các dòng trong bảng kết quả |
| `**params` | Tham số đưa thẳng cho `searchFn` (`beamWidth`, `maxRounds`, `weights`...) |

### Mỗi lần chạy ghi lại gì

| Cột | Ý nghĩa |
|---|---|
| `label` | Tên cấu hình |
| `seed` | Seed của lần chạy này |
| `score` | Điểm phương án tốt nhất tìm được |
| `numStations` | Phương án đó có mấy trạm |
| `rounds` | Chạy hết bao nhiêu vòng mới dừng |
| `seconds` | Thời gian chạy, dùng cho đồ thị đánh đổi ở Khối 6 |




In [ ]:
import time

def runExperiment(algorithm,label, numSeeds=20,**params):

    rows = []

    for seed in range(numSeeds):
        start = time.perf_counter()
        bestState, bestScore, history = algorithm(seed=seed, **params)
        seconds = time.perf_counter() - start
        
        numRuns = len({row["restart"] for row in history}) if "restart" in history[0] else 1
        
        rows.append({
            "label": label,
            "seed": seed,
            "score": bestScore,
            "numStations": len(bestState),
            "rounds": len(history) - numRuns,
            "seconds": seconds,
        })
        
    return pd.DataFrame(rows)


def summarize(df):
    return (df.groupby("label", sort=False)
              .agg(scoreMean=("score", "mean"),
                   scoreStd=("score", "std"),
                   scoreMax=("score", "max"),
                   stationsMean=("numStations", "mean"),
                   roundsMean=("rounds", "mean"),
                   secondsMean=("seconds", "mean"))
              .round(4)
              .reset_index())

## 5.2 Quét beamWidth

Chạy `localBeamSearch` với `beamWidth = 1, 3, 5, 10, 20`, mỗi giá trị nhiều seed.

Mục đích xem hai thứ:

- Beam rộng hơn có cho điểm cao hơn không
- Phải trả bao nhiêu thời gian

`beamWidth = 1` chính là Hill Climbing, dùng làm mốc so sánh.

Chạy một lần rồi lưu ra `results/beam_width.csv`. Khối 6 đọc file đó, khỏi chạy lại.

> Chạy đủ 20 seed với `maxRounds=50` mất khoảng 4 tiếng. Nên chạy nháp
> `numSeeds=10, maxRounds=30` trước để dựng biểu đồ.

In [ ]:
beamWidths = [1, 3, 5, 10]

# biến cờ đặt để chạy
RUN_EXPERIMENTS = False    

if RUN_EXPERIMENTS:
    frames = []
    for k in beamWidths:
        start = time.perf_counter()
        frames.append(runExperiment(localBeamSearch, label=f"k={k}",
                                    numSeeds=5, beamWidth=k, maxRounds=20))
        print(f"beamWidth={k:2d} xong sau {time.perf_counter() - start:6.1f}s")
    dfBeam = pd.concat(frames, ignore_index=True)
    dfBeam.to_csv("../results/beam_width.csv", index=False)
else:
    dfBeam = pd.read_csv("../results/beam_width.csv")

summarize(dfBeam)


## 5.3 So sánh các thuật toán

Chạy 4 thuật toán trên cùng dữ liệu, cùng số seed, rồi so điểm và thời gian.

| Tên | Thực chất là |
|---|---|
| Hill Climbing | `localBeamSearch` với `beamWidth=1` |
| Local Beam Search | `localBeamSearch` với `beamWidth=5` |
| Stochastic Beam | `stochasticLocalBeamSearch` với `beamWidth=5` |
| Random Restart | `randomRestartBeamSearch`, 4 lần x 5 vòng |

Random Restart chạy 4 lần nên mỗi lần chỉ cho 5 vòng, để tổng công sức bằng
các thuật toán kia (1 lần x 20 vòng). So mà không cân ngân sách thì vô nghĩa.

Lưu ra `results/algorithms.csv`.

In [ ]:
RUN_EXPERIMENTS = False

algorithms = [
    ("Hill Climbing",   localBeamSearch,           dict(beamWidth=1, maxRounds=20)),
    ("Local Beam",      localBeamSearch,           dict(beamWidth=5, maxRounds=20)),
    ("Stochastic Beam", stochasticLocalBeamSearch, dict(beamWidth=5, maxRounds=20)),
    ("Random Restart",  randomRestartBeamSearch,   dict(beamWidth=5, maxRounds=5, numRestarts=4)),
]

if RUN_EXPERIMENTS:
    frames = []
    for name, algorithm, params in algorithms:
        start = time.perf_counter()
        frames.append(runExperiment(algorithm, label=name, numSeeds=5, **params))
        print(f"{name:16} xong sau {time.perf_counter() - start:6.1f}s")
    dfAlgo = pd.concat(frames, ignore_index=True)
    dfAlgo.to_csv("../results/algorithms.csv", index=False)
else:
    dfAlgo = pd.read_csv("../results/algorithms.csv")

summarize(dfAlgo)

## 5.4 Phân tích độ nhạy trọng số

Giữ nguyên thuật toán, chỉ đổi trọng số chi phí trong `objective`:

In [ ]:
RUN_EXPERIMENTS = True

costWeights = [0.05, 0.1, 0.3, 0.5]

if RUN_EXPERIMENTS:
    frames = []
    for cost in costWeights:
        weights = {"pop": 1.0, "cost": cost, "penalty": 5.0}
        start = time.perf_counter()
        frames.append(runExperiment(localBeamSearch, label=f"cost={cost}",
                                    numSeeds=5, beamWidth=5, maxRounds=20,
                                    weights=weights))  
        print(f"cost={cost:<5} xong sau {time.perf_counter() - start:6.1f}s")

    dfWeights = pd.concat(frames, ignore_index=True)
    dfWeights.to_csv("../results/weights.csv", index=False)
else:
        dfWeights = pd.read_csv("../results/weights.csv")

summarize(dfWeights)

## 5.5 Lưu và gom kết quả

Ba mục trên đã tự lưu CSV vào `results/` ngay sau khi chạy xong.


In [ ]:
resultFiles = {
    "Quét beamWidth":     "../results/beam_width.csv",
    "So sánh thuật toán": "../results/algorithms.csv",
    "Độ nhạy trọng số":   "../results/weights.csv",
}

results = {}
for name, path in resultFiles.items():
    results[name] = pd.read_csv(path)
    print(f"{name:22} {len(results[name]):3d} lần chạy")

dfBeam    = results["Quét beamWidth"]
dfAlgo    = results["So sánh thuật toán"]
dfWeights = results["Độ nhạy trọng số"]

for name, df in results.items():
    print(f"\n=== {name} ===")
    display(summarize(df))